In [ ]:

!npm install -g localtunnel

from flask import Flask, render_template_string, request
from google.colab import output
import threading
import time

app = Flask(__name__)


storage = {
    'root': None,
    'original_text': ""
}


def build_tree(text):
    freq = {}
    order = []
    for char in text:
        if char not in freq:
            freq[char] = 0
            order.append(char)
        freq[char] += 1

    nodes = [{'char': char, 'freq': freq[char], 'left': None, 'right': None} for char in order]
    while len(nodes) > 1:
        nodes.sort(key=lambda x: x['freq'])
        left = nodes.pop(0)
        right = nodes.pop(0)
        merged = {'char': None, 'freq': left['freq'] + right['freq'], 'left': left, 'right': right}
        nodes.append(merged)
    return nodes[0] if nodes else None

def get_codes(node, current_code="", codes=None):
    if codes is None: codes = {}
    if node['char'] is not None:
        codes[node['char']] = current_code
        return codes
    get_codes(node['left'], current_code + "0", codes)
    get_codes(node['right'], current_code + "1", codes)
    return codes

def decode_huffman(binary, root):
    if not root: return "No Tree Found! Compress something first."
    decoded_text = ""
    curr = root
    for bit in binary:
        if bit == '0': curr = curr['left']
        else: curr = curr['right']
        if curr['char'] is not None:
            decoded_text += curr['char']
            curr = root
    return decoded_text


html_template = """
<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
    <meta charset="UTF-8">
    <title>Huffman Expert Tool</title>
    <style>
        body { font-family: 'Segoe UI', sans-serif; background-color: #f4f7f6; padding: 20px; }
        h1 { text-align: center; color: #2c3e50; margin-bottom: 30px; }
        .main-wrapper {
            display: flex;
            justify-content: center;
            align-items: flex-start;
            gap: 30px;
            flex-wrap: nowrap; /* Forces side-by-side */
            max-width: 1000px;
            margin: auto;
        }
        .card {
            background: white;
            padding: 25px;
            border-radius: 15px;
            box-shadow: 0 10px 20px rgba(0,0,0,0.05);
            flex: 1; /* Equal width cards */
            min-width: 320px;
        }
        .compress-card { border-top: 6px solid #3498db; }
        .decompress-card { border-top: 6px solid #2ecc71; }
        h2 { color: #34495e; font-size: 22px; margin-top: 0; }
        input { width: 90%; padding: 12px; margin: 15px 0; border: 1px solid #ddd; border-radius: 8px; font-size: 15px; }
        button { width: 100%; padding: 12px; border: none; border-radius: 8px; cursor: pointer; font-weight: bold; color: white; transition: 0.3s; font-size: 16px; }
        .btn-compress { background-color: #3498db; }
        .btn-compress:hover { background-color: #2980b9; }
        .btn-decompress { background-color: #2ecc71; }
        .btn-decompress:hover { background-color: #27ae60; }
        .results { margin-top: 20px; background: #f9f9f9; padding: 15px; border-radius: 8px; border-left: 4px solid #ddd; }
        .label { font-weight: bold; display: block; margin-bottom: 5px; color: #7f8c8d; }
        .val { font-weight: bold; color: #e74c3c; word-break: break-all; font-family: monospace; }
        .ratio-badge { background: #34495e; color: white; padding: 5px 12px; border-radius: 20px; display: inline-block; margin-top: 10px; font-size: 13px; }
    </style>
</head>
<body>
    <h1>Huffman Encoder & Decoder</h1>
    <div class="main-wrapper">

        <div class="card compress-card">
            <h2>Compression</h2>
            <form method="POST">
                <input type="hidden" name="action" value="compress">
                <input type="text" name="input_text" placeholder="Enter text (e.g., hello)" required>
                <button type="submit" class="btn-compress">Compress & Build Tree</button>
            </form>
            {% if comp %}
            <div class="results">
                <span class="label">Binary Output:</span>
                <span class="val">{{ comp.encoded }}</span><br>
                <div class="ratio-badge">Ratio: {{ comp.ratio }} : 1</div>
            </div>
            {% endif %}
        </div>

        <div class="card decompress-card">
            <h2>Decompression</h2>
            <form method="POST">
                <input type="hidden" name="action" value="decompress">
                <input type="text" name="input_bin" placeholder="Enter binary code..." required>
                <button type="submit" class="btn-decompress">Decompress to Text</button>
            </form>
            {% if decomp %}
            <div class="results">
                <span class="label">Decoded Original Text:</span>
                <span class="val" style="color:#2ecc71;">{{ decomp.text }}</span><br>
                <div class="ratio-badge">Verification: Match OK</div>
            </div>
            {% endif %}
        </div>

    </div>
</body>
</html>
"""

@app.route('/', methods=['GET', 'POST'])
def index():
    comp = None
    decomp = None
    if request.method == 'POST':
        action = request.form.get('action')

        if action == 'compress':
            text = request.form.get('input_text')
            storage['root'] = build_tree(text)
            codes = get_codes(storage['root'])
            encoded = "".join(codes[char] for char in text)
            ratio = round((len(text)*8) / len(encoded), 2) if len(encoded) > 0 else 0
            comp = {"encoded": encoded, "ratio": ratio}

        elif action == 'decompress':
            binary = request.form.get('input_bin')
            decoded = decode_huffman(binary, storage['root'])
            decomp = {"text": decoded}

    return render_template_string(html_template, comp=comp, decomp=decomp)


def run_app():
    app.run(port=8000)

print("\n--- TEAM ACCESS INFO ---")
print("Your Public IP is:", end=" ")
!curl ipv4.icanhazip.com
threading.Thread(target=run_app).start()
time.sleep(2)
!npx localtunnel --port 8000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 4s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋
--- TEAM ACCESS INFO ---
Your Public IP is: 34.24.113.167
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8000
INFO:werkzeug:Press CTRL+C to quit


⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://five-snails-nail.loca.lt


INFO:werkzeug:127.0.0.1 - - [30/Apr/2026 17:41:52] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [30/Apr/2026 17:41:53] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [30/Apr/2026 17:41:59] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [30/Apr/2026 17:42:06] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [30/Apr/2026 17:42:18] "POST / HTTP/1.1" 200 -
